# Title & Introduction 

# Notebook 4: Streamlit Deployment

## Objective
Deploy the calibrated Logistic Regression model as an interactive web application using Streamlit.

## Deliverables
1. Streamlit web app (`app.py`)
2. Requirements file for dependencies
3. Deployment instructions

## Model Used
- **File:** `outputs/models/logistic_regression_calibrated.pkl`
- **Type:** Calibrated Logistic Regression with StandardScaler
- **Features:** 20 numeric features
- **Performance:** PR AUC 0.9655, Perfect calibration (95.0% predicted vs 95.0% actual)

# Setup

In [1]:
import os

# Verify project structure
project_root = os.path.dirname(os.getcwd())
print(f"Project root: {project_root}")

# Check if streamlit_app folder exists
streamlit_path = os.path.join(project_root, 'streamlit_app')
if os.path.exists(streamlit_path):
    print(f"streamlit_app folder exists")
else:
    print(f"streamlit_app folder not found")

Project root: /Users/reynoldtakurachoruma/Desktop/Olist-Customer-Dropoff-Prediction
streamlit_app folder exists


# Create Folder Structure 

## 1. Create Streamlit App Folder

Creating the folder structure for the web application.

In [2]:
import os

# Create streamlit_app folder if it doesn't exist
streamlit_path = '../streamlit_app'
os.makedirs(streamlit_path, exist_ok=True)

print(f"Created folder: {streamlit_path}")

Created folder: ../streamlit_app


## 2. Create Streamlit Web Application

The app provides:
- **Interactive customer profile input** (delivery, economics, customer attributes)
- **Real-time drop-off risk prediction** using the calibrated model
- **Personalized recommendations** based on feature importance
- **ROI calculator** to assess intervention cost-effectiveness

The app uses only the 20 features the calibrated model expects (excluding payment type and state features that were not used during training).

# App.py

In [5]:
if st.button("Predict Drop-off Risk", type="primary"):

    is_late_delivery = int(delivery_delay > 0)
    is_very_late = int(delivery_delay > 10)
    is_early_delivery = int(delivery_delay < 0)
    is_high_freight = int(freight_pct > 20)
    cluster_0 = int(cluster == "Budget Shoppers (0)")
    cluster_1 = int(cluster == "High Risk (1)")

    features = pd.DataFrame({
        'delivery_delay': [float(delivery_delay)],
        'is_late_delivery': [int(is_late_delivery)],
        'is_very_late': [int(is_very_late)],
        'is_early_delivery': [int(is_early_delivery)],
        'freight_pct': [float(freight_pct)],
        'is_high_freight': [int(is_high_freight)],
        'num_items': [int(num_items)],
        'price_per_item': [float(price_per_item)],
        'uses_installments': [int(uses_installments)],
        'is_southeast': [int(is_southeast)],
        'is_repeatable_category': [int(is_repeatable_category)],
        'is_heavy_product': [int(is_heavy_product)],
        'has_comment': [int(has_comment)],
        'purchase_month': [int(purchase_month)],
        'purchase_day_of_week': [int(purchase_day_of_week)],
        'is_weekend': [int(is_weekend)],
        'is_holiday_season': [int(is_holiday_season)],
        'days_to_delivery': [int(days_to_delivery)],
        'cluster_0': [int(cluster_0)],
        'cluster_1': [int(cluster_1)]
    })

    try:
        prediction_proba = model.predict_proba(features)[0]
        drop_off_prob = prediction_proba[1] * 100
        retention_prob = prediction_proba[0] * 100

        if drop_off_prob >= 90:
            risk_level = "CRITICAL RISK"
            risk_color = "red"
        elif drop_off_prob >= 80:
            risk_level = "HIGH RISK"
            risk_color = "orange"
        elif drop_off_prob >= 60:
            risk_level = "MEDIUM RISK"
            risk_color = "blue"
        else:
            risk_level = "LOW RISK"
            risk_color = "green"

        st.divider()
        st.header("Prediction Results")

        metric_col1, metric_col2, metric_col3 = st.columns(3)

        with metric_col1:
            st.metric(
                "Drop-off Probability",
                f"{drop_off_prob:.1f}%",
                delta=f"{drop_off_prob - 95:.1f}% vs baseline",
                delta_color="inverse"
            )

        with metric_col2:
            st.metric(
                "Retention Probability",
                f"{retention_prob:.1f}%",
                delta=f"{retention_prob - 5:.1f}% vs baseline"
            )

        with metric_col3:
            if risk_color == "red":
                st.error(f"🔴 {risk_level}")
            elif risk_color == "orange":
                st.warning(f"🟠 {risk_level}")
            elif risk_color == "blue":
                st.info(f"🟡 {risk_level}")
            else:
                st.success(f"🟢 {risk_level}")

        st.subheader("Personalized Recommendations")

        recommendations = []

        if drop_off_prob >= 90:
            recommendations.append("**URGENT:** High-risk customer - activate retention protocol immediately")
        if is_high_freight:
            recommendations.append("**High shipping cost** - Consider free shipping offer")
        if not is_repeatable_category:
            recommendations.append("**Non-repeatable product** - Cross-sell to recurring categories")
        if not uses_installments:
            recommendations.append("**Promote installment payments** - linked to higher retention")
        if is_holiday_season:
            recommendations.append("**Holiday purchase** - higher likelihood to return!")
        else:
            recommendations.append("**Consider seasonal promotion** to re-engage")
        if is_late_delivery:
            recommendations.append("**Late delivery** - Issue apology credit or compensation")
        if not is_southeast:
            recommendations.append("**Non-Southeast customer** - Extra attention needed")

        for rec in recommendations:
            st.markdown(f"- {rec}")

        st.subheader("Retention ROI Estimate")

        roi_col1, roi_col2 = st.columns(2)

        intervention_cost = 15
        intervention_success_rate = 0.30
        customer_ltv = 160

        with roi_col1:
            st.markdown(f"**Intervention Cost:** R$ {intervention_cost}")
            st.markdown(f"**Intervention Success Rate:** {intervention_success_rate:.0%}")
            st.markdown(f"**Customer LTV:** R$ {customer_ltv}")

        with roi_col2:
            expected_value = (retention_prob / 100) * intervention_success_rate * customer_ltv - intervention_cost
            roi = ((expected_value + intervention_cost) / intervention_cost - 1) * 100

            st.metric("Expected Value", f"R$ {expected_value:.2f}")
            st.metric("ROI", f"{roi:.1f}%")

            if expected_value > 0:
                st.success("Intervention recommended")
            else:
                st.warning("Intervention not cost-effective")

    except Exception as e:
        st.error(f"Prediction error: {e}")
        st.error("Please check inputs and try again")

2026-04-04 16:23:11.636 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 16:23:11.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 16:23:11.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 16:23:11.690 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-04 16:23:11.694 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## 3. Create Requirements File

Dependencies needed to run the Streamlit app.

In [6]:
%%writefile ../streamlit_app/requirements.txt
streamlit==1.31.0
pandas==2.1.4
scikit-learn==1.3.2

Overwriting ../streamlit_app/requirements.txt


## 4. Testing Instructions

To test the app locally:

1. **Navigate to streamlit_app folder:**
```bash
   cd streamlit_app
```

2. **Install dependencies:**
```bash
   pip install -r requirements.txt
```

3. **Run the app:**
```bash
   streamlit run app.py
```

4. **Test with different customer profiles:**
   - **Best case:** Holiday season, repeatable category, early delivery, low freight
   - **Worst case:** Late delivery, high freight, non-repeatable, weekend purchase

The app should show:
- Best case: ~19% drop-off (81% retention)
- Worst case: ~96% drop-off (4% retention)

## 5. Deployment Options

### Option 1: Streamlit Community Cloud (Recommended)
1. Push project to GitHub
2. Go to [share.streamlit.io](https://share.streamlit.io)
3. Connect GitHub repository
4. Deploy `streamlit_app/app.py`

### Option 2: Local Deployment
- Run locally using instructions above
- Share via ngrok for temporary public access

### Option 3: Docker Deployment
- Create Dockerfile for containerized deployment
- Deploy to cloud platforms (AWS, GCP, Azure)

## Project Complete! ✅

The Streamlit app is ready for:
- ✅ Local testing
- ✅ Live demonstration
- ✅ Cloud deployment
- ✅ Portfolio showcase